# HF 두 층 재구성과 수정주가 실측 — 2026-09-02

**오늘 한 일 세 가지** — 전부 실측과 음성 대조군을 붙여 확인했다.

| 작업 | 결과 |
|---|---|
| **A. HF `alphastack-dart` 2층 재구성** | 루트 벌크 43개 → `bulk/` 이동(재업로드 없음) · `pit/` 11개(662,933행) 업로드 · 값 단위 대조 11/11 통과 |
| **B. 수정주가 검증 (#51)** | FDR 수정주가가 분할 갭을 정확히 고침 — 방향 확정: **`adj_close` 신설** (v8 구현) |
| **C. Kronos zero-shot 스모크** | Colab T4 에서 512봉→30봉 예측 동작 · 피크 VRAM 0.13GB |

왜 두 층인가: 기존 벌크 parquet 에는 **공시 접수일(`rcept_dt`)이 없다.** 결산기에 값을
붙이면 석 달치 미래가 학습에 들어가고 예외는 나지 않는다. 접수일이 있는 API 재수집분을
`pit/`(point-in-time) 층으로 올려 학습 정본으로 삼고, 벌크는 참조용으로 남긴다.


## A. 반출본 값 단위 대조 — 행 수가 아니라 모든 칸

행 수 검증은 덮어쓰기를 못 잡는다(팀 실측 교훈). 파일과 DB 를 전 컬럼으로 정렬한 뒤
`DataFrame.equals` 로 **모든 칸**을 비교하고, 검증식 자체가 살아 있는지 음성 대조군
두 벌(값 1건 변조 · 연도 어긋난 쌍)로 확인한다.

DB 는 수집 세션과 병행 중이라 **읽기 전용(`mode=ro`)으로만** 연다.


In [1]:
%run ../../scripts/_verify_pit_export.py --db c:/Users/kik32/workspace/EST-Camp-AI-Quant/team_project/Alpha_Stack/data/krx_cache.db --outbox ../../data/outbox/dart_pit_20260902

── 값 단위 대조 — 11개 파일 ──


  ✅ 2015_사업보고서.parquet           파일  46,428행 vs DB  46,428행 → equals=True


  ✅ 2016_사업보고서.parquet           파일  46,817행 vs DB  46,817행 → equals=True


  ✅ 2017_사업보고서.parquet           파일  49,127행 vs DB  49,127행 → equals=True


  ✅ 2018_사업보고서.parquet           파일  54,257행 vs DB  54,257행 → equals=True


...

  ✅ 2022_사업보고서.parquet           파일  60,831행 vs DB  60,831행 → equals=True


  ✅ 2023_사업보고서.parquet           파일  72,378행 vs DB  72,378행 → equals=True


  ✅ 2024_사업보고서.parquet           파일  78,098행 vs DB  78,098행 → equals=True


  ✅ 2025_사업보고서.parquet           파일  80,967행 vs DB  80,967행 → equals=True

── 음성 대조군 ──
  ✅ 값 1건 변조(bsns_year) → equals=False 가 나온다: True


  ✅ 2015 파일 vs 2016 DB → equals=False 가 나온다: True

✅ 전부 통과


## A-2. HF 서버에 실제로 올라간 것

이동은 `CommitOperationCopy` 라 LFS 포인터만 복사된다 — 443MB 를 다시 올리지 않았다.
기존 루트 README(벌크 상세 문서)는 `bulk/README.md` 로 보존했다.

⚠️ 병행 세션과의 교차: 수집 세션이 오늘 11:59 에 올렸던 `v6_with_receipt_date/`
(dev/holdout 분할본)를 12:21 에 스스로 지웠다 — "pit/ 층과 같은 자료라 한 벌만 남긴다".
그래서 아래 실측에서 v6 는 0개이고, 최종 구조는 원설계(PR #62)와 같은 **2층**이다.


In [2]:
import hashlib, json
from huggingface_hub import HfApi, hf_hub_download

REPO = "qurious-quant/alphastack-dart"
files = HfApi().list_repo_files(REPO, repo_type="dataset")
for prefix in ("bulk/", "pit/", "v6_with_receipt_date/"):
    n = sum(f.startswith(prefix) for f in files)
    print(f"  {prefix:<24} {n}개")
root_parquet = [f for f in files if f.endswith(".parquet") and "/" not in f]
print(f"  루트에 남은 parquet    {len(root_parquet)}개 (0이어야 한다)")

# 서버 파일이 우리가 만든 그 파일인가 — MANIFEST 의 SHA-256 과 대조
m = json.load(open(hf_hub_download(REPO, "MANIFEST.json", repo_type="dataset"), encoding="utf-8"))
target = m["files"][0]
h = hashlib.sha256(open(hf_hub_download(REPO, target["path"], repo_type="dataset"), "rb").read()).hexdigest()
print(f"  SHA-256 대조 {target['path']}: {'✅ 일치' if h == target['sha256'] else '🔴 불일치'}")

  bulk/                    44개
  pit/                     11개
  v6_with_receipt_date/    0개
  루트에 남은 parquet    0개 (0이어야 한다)


MANIFEST.json: 0.00B [00:00, ?B/s]

C:\Users\kik32\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kik32\.cache\huggingface\hub\datasets--qurious-quant--alphastack-dart. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


(…)97%85%EB%B3%B4%EA%B3%A0%EC%84%9C.parquet:   0%|          | 0.00/989k [00:00<?, ?B/s]

  SHA-256 대조 pit/2015_사업보고서.parquet: ✅ 일치


## B. 수정주가 검증 (#51) — FDR 이 분할 갭을 실제로 고치는가

`daily_price.close` 는 분할 미조정 원가격이다. 삼성전자 2018-05-04(50:1)를 close 로
계산하면 −98.04% 로 읽힌다. FinanceDataReader(MIT) 수정주가로 갈아타기 전에 세 가지를 쟀다.

1. **분할일 갭** — FDR 갭이 KRX `change_rate`(이미 조정된 값)와 맞는가
2. **조정계수 역산** — 원주가÷FDR 비율이 갈리는 배수가 공지 비율(50:1, 5:1)과 맞는가
3. **음성 대조군** — 분할 없는 종목에서 비율이 1.0 으로 일정한가 (배수가 잡히면 검증식이 틀린 것)

분할일은 문서가 아니라 **DB 주식수 점프**(주식수 배율 × 가격 배율 ≈ 1)로 재확인한다.


In [3]:
%run ../../scripts/_probe_fdr_adjust.py --db c:/Users/kik32/workspace/EST-Camp-AI-Quant/team_project/Alpha_Stack/data/krx_cache.db


── 삼성전자(005930) 분할일 20180504 ──
  DB 재확인   주식수 50.000배 × 가격 0.01958배 = 0.979 → 분할일 맞음: True
  원주가 갭   -98.04%  (이걸 수익률로 쓰면 안 된다)
  FDR 갭     -2.08%  vs KRX change_rate -2.08% → ✅ 일치
  조정계수    50.000  vs 공지 비율 50:1 → ✅ 일치

── NAVER(035420) 분할일 20181012 ──
  DB 재확인   주식수 5.000배 × 가격 0.20170배 = 1.009 → 분할일 맞음: True
  원주가 갭   -79.83%  (이걸 수익률로 쓰면 안 된다)
  FDR 갭     +0.71%  vs KRX change_rate +0.71% → ✅ 일치
  조정계수    4.993  vs 공지 비율 5:1 → ✅ 일치



── 카카오(035720) 분할일 20210415 ──
  DB 재확인   주식수 5.000배 × 가격 0.21595배 = 1.080 → 분할일 맞음: True
  원주가 갭   -78.41%  (이걸 수익률로 쓰면 안 된다)
  FDR 갭     +7.59%  vs KRX change_rate +7.59% → ✅ 일치
  조정계수    4.982  vs 공지 비율 5:1 → ✅ 일치

── SK하이닉스(000660) 음성 대조군 · 2,863일 ──
  분할 판정식에 걸린 날: 0일 (0이어야 대조군 자격)
  원주가/FDR 비율  최소 1.0000 · 최대 1.0000 (1.0 근처로 일정해야 한다)
  → ✅ 비율 일정 — 검증식이 분할을 없는 곳에서 만들지 않는다



── 현대차(005380) 음성 대조군 · 2,863일 ──
  분할 판정식에 걸린 날: 0일 (0이어야 대조군 자격)
  원주가/FDR 비율  최소 1.0000 · 최대 1.0000 (1.0 근처로 일정해야 한다)
  → ✅ 비율 일정 — 검증식이 분할을 없는 곳에서 만들지 않는다

✅ 전부 통과 — 분할 3/3 · 대조군 2/2


### B-2. 상장폐지 종목 커버리지 — 생존 편향 구멍이 없는가

유니버스에는 폐지 종목이 들어 있다. FDR 이 폐지 종목의 과거 구간을 못 주면
수정주가 층에 구멍이 난다. 20200102 에 있다가 20260820 에 없는 종목으로 잰다.


In [4]:
import sqlite3
import FinanceDataReader as fdr

conn = sqlite3.connect("file:c:/Users/kik32/workspace/EST-Camp-AI-Quant/team_project/Alpha_Stack/data/krx_cache.db?mode=ro", uri=True)
old = {r[0]: r[1] for r in conn.execute(
    "SELECT code, name FROM daily_price WHERE bas_dd = '20200102'")}
recent = {r[0] for r in conn.execute(
    "SELECT code FROM daily_price WHERE bas_dd = '20260820'")}
conn.close()

gone = sorted(set(old) - recent)
print(f"20200102 존재 {len(old):,}종 · 20260820 존재 {len(recent):,}종 · 사라진 종목 {len(gone):,}종")
for code in gone[:5]:
    df = fdr.DataReader(code, "2019-01-01", "2020-12-31")
    print(f"  {code} {old[code]:<12s} FDR {len(df):>4,}행")
print("  (대조) 005930", len(fdr.DataReader("005930", "2019-01-01", "2020-12-31")), "행 — 같은 구간 정상")

20200102 존재 2,324종 · 20260820 존재 2,763종 · 사라진 종목 239종
  000060 메리츠화재        FDR  494행


  000075 삼양홀딩스우       FDR  494행


  000547 흥국화재2우B      FDR  494행
  000885 한화우          FDR  494행


  000995 DB하이텍1우      FDR  494행


  (대조) 005930 494 행 — 같은 구간 정상


### B-3. 결정 — `close` 덮어쓰기가 아니라 수정 OHLC 4칸 신설

| 근거 | 내용 |
|---|---|
| 시총 정합 | `market_cap = close × listed_shares` 는 **원가격**이어야 맞다 — 덮으면 깨진다 |
| 소급 재계산 | 수정주가는 다음 분할 때 **과거 전체가 다시 바뀐다** — append-only 워터마크 반입과 충돌 |
| Kronos 입력 | OHLCV 6채널이라 **수정 OHLC 전부** 필요 — FDR 이 다 준다 |

값 소스는 **FDR + 자체 교차검증**: 적재 시 (a) FDR 일간 갭 ↔ KRX `change_rate`,
(b) 분할일 조정계수 ↔ 주식수 역산 배율을 통과한 값만 싣는다 (위 검증식 재사용).
구현은 **마이그레이션 v8 세션** — 지금은 수집 세션 병행 중이라 마이그레이션 금지.
기록: [이슈 #51 코멘트](https://github.com/devlee328288/Alpha_Stack/issues/51#issuecomment-5503867905)


## C. Kronos zero-shot 스모크 — Colab T4 실측 기록

colab-mcp 는 stateless 실행기로만 쓴다(셀 편집이 유지되지 않는 것이 실측됨) —
**이 노트북이 정본**이고, Colab 에는 실행할 코드만 던져 출력을 회수했다.

### 실행한 셀 1 — GPU 확인

```python
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True, timeout=30)
print("nvidia-smi:", out.stdout.strip())
import torch
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
```

```
nvidia-smi: Tesla T4, 15360 MiB
torch: 2.11.0+cu128 | cuda available: True
device: Tesla T4
```

### 실행한 셀 2 — 설치

```python
!git clone --depth 1 https://github.com/shiyu-coder/Kronos.git /content/Kronos
%pip install -q einops finance-datareader
import sys; sys.path.insert(0, "/content/Kronos")
from model import Kronos, KronosTokenizer, KronosPredictor
```

### 실행한 셀 3 — 예측 (삼성전자 · FDR 수정 OHLCV · amount 는 close×volume 근사)

```python
df = fdr.DataReader("005930", "2024-01-01")
# ... (OHLCV 6채널 구성, 컨텍스트 512봉 + 검증 30봉)
tokenizer = KronosTokenizer.from_pretrained("NeoQuasar/Kronos-Tokenizer-base")
model = Kronos.from_pretrained("NeoQuasar/Kronos-small")
predictor = KronosPredictor(model, tokenizer, device="cuda:0", max_context=512)
pred_df = predictor.predict(df=x_df, x_timestamp=x_ts, y_timestamp=y_ts,
                            pred_len=30, T=1.0, top_p=0.9, sample_count=1)
```

```
컨텍스트 2024-06-12 ~ 2026-07-21 (512봉)
예측 구간 2026-07-22 ~ 2026-09-02 (30봉)
모델 로드 6.6초 · 예측 1.5초 · 피크 VRAM 0.13 GB
예측 close 처음 3봉: [259249.0, 228716.0, 213219.0]
실제 close 처음 3봉: [260500, 270000, 249500]
30봉 close MAE 109,696원 vs 나이브(마지막값 유지) 14,583원
(zero-shot 1표본 — 품질 평가가 아니라 파이프라인 동작 확인)
```

### 읽는 법

- **파이프라인은 완주했다** — 로드 6.6초 · 예측 1.5초 · VRAM 0.13GB. T4(15.4GB)에 여유가 크다.
  fine-tune VRAM 은 아직 미측정이지만 zero-shot 기준으로는 병목이 없다.
- **zero-shot 1표본 품질은 나이브보다 나쁘다** (MAE 109,696원 vs 14,583원).
  T=1.0 · sample_count=1 의 확률 샘플링 1회라 예상 범위다 — 품질 판단은
  fine-tune + 다표본 평가에서 한다. 여기서 결론 내지 않는다.
- amount 채널은 close×volume 근사를 썼다 — 본 학습에서는 DB `value`(실측 거래대금)를 쓴다.


## 남은 것

- `adj_open/high/low/close` 4칸 적재 — **마이그레이션 v8 세션** (#51 은 그때 닫는다)
- `HOLDOUT_START` 20250901 제안 — 팀 결정 대기 (#60 닫힘 · 제안만 남음, 정본은 `evaluation/horizon.py`)
- Kronos fine-tune VRAM · 다표본 평가 — 수정주가 4칸 적재 후
- 루트 README 의 v6 행 제거 커밋 — v6 층이 지워져 안내가 실제와 어긋난 상태 (사용자 확인 후 push)
